# Taller 2 — Diseño del workflow de monitoreo de Toyota Corolla
**Equipo:** Jair Acevedo · Dayanna Reyes  **Repo:** https://github.com/jairesacevedo/busqueda-corolla
**Estado:** propuesta para revisar (9-nov-2026) · **Entrega:** 23-nov-2026, 11:59 p. m. (Bloque Neón)

Este notebook tiene dos partes:
1. **El diseño** (texto): qué construimos, cómo y por qué, punto por punto del enunciado.
2. **Demostración en vivo** (código): consulta Carroya ahora mismo para que veamos los datos reales y la regla de gangas. Requiere `pip install requests pandas matplotlib`.

> Nada de esto es todavía el pipeline final: es para ponernos de acuerdo antes de construir.

## Calendario del taller
| Fecha | Qué pasa |
|---|---|
| 10–11 nov | Construir recolección, consolidación y logs |
| 12–13 nov | Análisis de gangas, plantilla del correo, workflows de GitHub Actions |
| 14–15 nov | **Prueba completa obligatoria** (datos y correos marcados como PRUEBA) |
| **16–21 nov** | **Periodo evaluado:** recolección 8:00 a. m., correo 11:00 a. m. (hora Colombia) |
| 22–23 nov | Documento de 1 página, README, zip y entrega |

**Peso de cada punto:** recolección 20 % · base 15 % · actualización 10 % · automatización 10 % · logs 10 % · gangas 20 % · correo 5 % · entrega 10 %.

## 0. Arquitectura general

```
            GitHub Actions — 8:00 a. m. (America/Bogota)                    GitHub Actions — 11:00 a. m.
 ┌───────────┐   ┌───────────┐   ┌─────────────┐   ┌──────────┐   ┌────────┐      ┌───────────────┐
 │ Búsqueda  │ → │ Descarga  │ → │Consolidación│ → │ Análisis │ → │ commit │ ···> │ Enviar correo │
 │ (3 págs.) │   │ (detalle) │   │  (SQLite)   │   │ (gangas) │   │al repo │      │ (Gmail SMTP)  │
 └───────────┘   └───────────┘   └─────────────┘   └──────────┘   └────────┘      └───────────────┘
        └───────────── cada etapa escribe una fila en logs/log_ejecuciones.csv ─────────────┘
```

- **Dónde corre:** GitHub Actions. **No necesita ningún computador encendido.** Ya comprobamos (9-nov) que los servidores de GitHub sí llegan a la API de Carroya.
- **Dónde se guarda:** en el mismo repositorio. Cada ejecución hace *commit* de los archivos del día, así el repo es a la vez almacenamiento y evidencia con fecha y hora.

```
busqueda-corolla/
├── src/corolla/            recolectar.py · consolidar.py · analizar.py · correo.py · logs.py
├── datos/diarios/AAAA-MM-DD/  enlaces_AAAA-MM-DD.csv · datos_AAAA-MM-DD.csv
├── datos/corolla.db        base consolidada (SQLite)
├── reportes/AAAA-MM-DD/    resumen.json · distribucion.png · gangas.csv
├── logs/log_ejecuciones.csv
├── pruebas/                igual que arriba, solo para pruebas (nunca se mezcla)
└── .github/workflows/      diario.yml (8:00) · correo.yml (11:00)
```

## 1. Cómo funciona Carroya (lo que encontramos)
La página web es una aplicación JavaScript (el HTML llega vacío), **pero los datos vienen de una API pública sin autenticación**. No necesitamos Selenium ni leer HTML:

| Para qué | Petición | Respuesta |
|---|---|---|
| Búsqueda | `POST .../find-filters` con marca, línea, rango de años y página | `count` = 60, `pages` = 3, 20 anuncios por página en `results.superHighlights` |
| Detalle de un anuncio | `GET .../find-vehicle-detail/{uuid}` | precio, versión, año, km, caja, combustible, ciudad, accesorios, fechas… |

- **Identificador estable:** el UUID de cada anuncio (ej. `d1bf9d95-155f-4610-ab51-e93207c59ff7`), que también está en su URL.
- **⚠️ Datos personales:** la API también trae nombre, correo, dirección y placa del vendedor → **no los guardamos** (lista blanca de campos).

Las dos celdas siguientes lo demuestran en vivo.

In [ ]:
import requests, pandas as pd, re, time

API = "https://carroya-habilitadora-api.avaldigitallabs.com"
HEADERS = {"Content-Type": "application/json",
           "User-Agent": "Mozilla/5.0 (taller-python-uniandes; monitoreo academico)",
           "Origin": "https://www.carroya.com", "Referer": "https://www.carroya.com/"}
LISTAS = ("superHighlights", "megaHighlights", "latestReleases")

def buscar_pagina(pagina: int) -> dict:
    """Pide una página de resultados de la búsqueda Toyota Corolla 2022-2026."""
    cuerpo = {"isReqForRedirect": False, "seoArray": [],
              "params": {"brand": "Toyota", "category": "Automóviles y Camionetas",
                         "model": "Corolla", "modelRange": "2022-2026", "page": str(pagina)}}
    r = requests.post(f"{API}/find-filters", json=cuerpo, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()

# Recorrer TODAS las páginas hasta que una llegue vacía
anuncios, pagina, total_declarado = [], 1, None
while True:
    j = buscar_pagina(pagina)
    total_declarado = j["count"]
    lote = [a for lista in LISTAS for a in (j["results"].get(lista) or [])]
    print(f"página {pagina}: {len(lote)} anuncios")
    if not lote:
        break
    for a in lote:
        a["_pagina"] = pagina
    anuncios += lote
    pagina += 1
    time.sleep(1)  # cortesía con el servidor

df = pd.DataFrame(anuncios).drop_duplicates("id")
completa = len(df) == total_declarado
print(f"\nTotal declarado: {total_declarado} · únicos recolectados: {len(df)} · búsqueda completa: {completa}")

In [ ]:
# Verificar criterios y quedarnos SOLO con los campos que necesitamos (lista blanca)
ok = df["brand"].eq("Toyota") & df["model"].eq("Corolla") & df["year"].between(2022, 2026)
print("Anuncios que cumplen los criterios:", ok.sum(), "de", len(df))

SUNROOF = re.compile(r"techo corredizo|techo panor[aá]mico|sunroof", re.I)
def sunroof(accesorios, comentarios):
    """'si' / 'no' / None (desconocido si el anuncio no trae lista de accesorios)."""
    texto = f"{accesorios or ''} {comentarios or ''}"
    if SUNROOF.search(texto):
        return "si"
    return "no" if accesorios else None

vista = pd.DataFrame({
    "id_anuncio": df["id"],
    "url": "https://www.carroya.com/detalle/usado/toyota/corolla/" + df["year"].astype(str) + "/x/" + df["id"],
    "anio": df["year"], "version": df["clinea2"], "precio": df["price"], "moneda": df["ctipomoneda"],
    "kilometraje": df["kilometers"], "transmision": df["ctipocaja"].replace("", None),
    "combustible": df["ccombustible"], "ciudad": df["city"].str.title(),
    "sunroof": [sunroof(a, c) for a, c in zip(df["caccesorios"], df["ccomentarios"])],
})[ok.values]
print(vista["sunroof"].fillna("desconocido").value_counts())
vista.head(10)

In [ ]:
# Detalle de UN anuncio (lo que el pipeline hará con cada uno)
uid = vista["id_anuncio"].iloc[0]
d = requests.get(f"{API}/find-vehicle-detail/{uid}", headers=HEADERS, timeout=30).json()["data"]
CAMPOS = ["id", "brand", "model", "year", "version", "price", "kilometers", "box", "fuel",
          "city", "color", "accessories", "sold", "fechapublicacion", "fechaactualizacion"]
{k: d.get(k) for k in CAMPOS}

## 2. Punto 1 — Archivos diarios (20 %)
Se guardan **dos archivos por día** en `datos/diarios/AAAA-MM-DD/`.

**`enlaces_AAAA-MM-DD.csv`** — una fila por anuncio, sin repetir:

| columna | ejemplo |
|---|---|
| id_anuncio | d1bf9d95-155f-… |
| url | https://www.carroya.com/detalle/usado/toyota/corolla/2022/villavicencio/… |
| pagina_busqueda | 1 |
| fecha_hora_descubrimiento | 2026-11-16T08:00:12-05:00 |
| estado_descarga | `sin intentar` / `descargado` / `fallido` |
| fecha_hora_ultimo_intento | *(vacío si sin intentar)* |
| motivo_error | *(solo si fallido)* |
| id_ejecucion | 20261116-0800 |

**`datos_AAAA-MM-DD.csv`** — una fila por anuncio, **incluidos los fallidos** (con campos del vehículo vacíos):
`id_anuncio, url, estado_descarga, fecha_hora_descarga, motivo_error, marca, linea, anio, version, precio, moneda, kilometraje, transmision, combustible, ciudad, sunroof, color, fecha_publicacion_carroya, fecha_actualizacion_carroya, vendido, id_ejecucion`

**Reglas:**
- **Búsqueda completa** = todas las páginas respondieron bien **y** los IDs únicos = `count`. Se usa en el punto 3.
- Descarga con **3 reintentos** con espera creciente; si falla, `fallido` + motivo (ej. `HTTP 503`, `timeout`).
- **Dato faltante = vacío.** Nunca se inventa ni se pone 0.
- **Sunroof:** `si` si aparece "techo corredizo/panorámico/sunroof"; `no` si hay lista de accesorios sin eso; **vacío** si el anuncio no trae accesorios (26 de 60 hoy).

## 3. Punto 2 — Base consolidada en SQLite (15 %)
**Identificador:** el UUID de Carroya → el mismo anuncio nunca se crea dos veces.

| Tabla | Llave | Qué guarda |
|---|---|---|
| `anuncios` | id_anuncio | ficha más reciente, `primera_vez_visto`, `ultima_vez_visto`, `disponibilidad_actual` |
| `observaciones` | (id_anuncio, fecha) | una fila por anuncio y día: disponibilidad, estado y hora de descarga, precio, km |
| `cambios` | (id_anuncio, fecha, campo) | valor anterior → valor nuevo (precio, km, versión…) |
| `archivos_procesados` | sha256 del archivo | qué archivos diarios ya se procesaron |

**Mismo vehículo en anuncios distintos (opcional):** se marca *posible duplicado* si coinciden año, versión, color, ciudad y el km difiere < 2 %. Solo se reporta, no se fusiona. Limitación: no tenemos la placa completa.

**¿Por qué SQLite y no otra cosa?**

| | **SQLite (elegida)** | CSV | Parquet |
|---|---|---|---|
| Consultas y cruces | SQL completo | cargar todo en pandas | pandas / duckdb |
| Evitar duplicados | PRIMARY KEY + UPSERT nativos | a mano | a mano |
| Varias tablas relacionadas | un solo archivo | un archivo por tabla | un archivo por tabla |
| Abrir sin programar | DB Browser | Excel ✅ | no |
| Ver cambios en git | binario | diff legible ✅ | binario |

Para compensar sus desventajas, cada día exportamos también las tablas a CSV.

## 4. Punto 3 — Funciones de actualización (10 %)
`consolidar(ruta_enlaces, ruta_datos)`:
1. Calcula el **sha256** del archivo → si ya se procesó, **no hace nada** (y lo registra).
2. Anuncio nuevo → INSERT. Anuncio existente → compara campo a campo y guarda diferencias en `cambios`.
3. **UPSERT** en `observaciones` por (id, fecha) → repetir el proceso no duplica.
4. Asigna disponibilidad a **todos** los anuncios conocidos:

| Situación | Disponibilidad |
|---|---|
| Apareció en la búsqueda (aunque falle el detalle) | `encontrado` |
| No apareció y la búsqueda fue **completa** | `no_encontrado` *(≠ vendido)* |
| No apareció y la búsqueda fue **incompleta** | `desconocido` |

**Prueba obligatoria:** procesar el mismo día dos veces debe dejar exactamente los mismos conteos.

## 5. Punto 4 — Automatización (10 %)
| Workflow | Cron (UTC) | Hora Colombia | Qué hace |
|---|---|---|---|
| `diario.yml` | `40 12 16-21 11 *` | 7:40 → espera a las **8:00:00** | búsqueda, descarga, consolidación, análisis, commit |
| `correo.yml` | `40 15 16-21 11 *` | 10:40 → espera a las **11:00:00** | lee el repo y envía el correo |

- **¿Por qué arrancar 20 min antes?** GitHub retrasa a veces los workflows programados 10–20 min; el script espera hasta la hora exacta.
- Ambos se pueden lanzar a mano con `modo=prueba` → escriben en `pruebas/` y el correo dice **[PRUEBA]**.
- **Condiciones:** ningún PC encendido. Necesita el repo activo, el *secret* con la contraseña de aplicación de Gmail y que la API de Carroya siga igual.

## 6. Punto 5 — Logs (10 %)
`logs/log_ejecuciones.csv`, una fila por **etapa** de cada ejecución:

`id_ejecucion, modo (real/prueba), etapa (recoleccion | descarga | consolidacion | analisis | correo), inicio, fin, duracion_s, estado (ok | parcial | error | omitido), enlaces_encontrados, anuncios_nuevos, descargas_ok, descargas_fallidas, descargas_pendientes, registros_agregados, registros_actualizados, registros_sin_cambio, anuncios_unicos_total, tamano_db_kb, busqueda_completa, errores, estado_correo`

**Zona horaria:** America/Bogota con desfase explícito en todos los archivos → `2026-11-16T08:00:12-05:00`.

## 7. Punto 6 — Precios y gangas (20 %)
**Distribución diaria:** una observación por anuncio (solo las descargadas ese día con precio válido) → histograma + boxplot por año, con n, mínimo, Q1, mediana, Q3, máximo.

**Precio inválido:** vacío, < $20 M, > $400 M o moneda ≠ pesos → se excluye y se cuenta.

**Método: comparables con jerarquía.** Para cada anuncio buscamos el grupo de referencia más específico con **al menos 5 comparables** (sin contarse a sí mismo):
1. mismo año + mismo combustible + misma banda de km (0–25k, 25–50k, 50–100k, >100k)
2. mismo año + mismo combustible
3. año ±1 + mismo combustible
4. si ninguno llega a 5 → **"datos insuficientes"** (se reporta)

**Regla de alerta:** `precio ≤ 0,85 × mediana del grupo` **y** `precio < Q1 del grupo`. Usamos la mediana porque no la mueven los extremos.

**¿Por qué estas variables?** El año y el km son los que más explican el precio de un usado; el combustible separa dos mercados distintos (52 híbridos vs. 8 a gasolina). Versión y sunroof se muestran en la explicación, pero no definen el grupo porque con ~60 anuncios los grupos quedarían de 1–2 carros.

**Limitaciones:** precio publicado ≠ precio de venta; puede haber errores de digitación del vendedor; el grupo de referencia cambia de un día a otro; pocos datos.

Solo se alertan en el correo los anuncios **nuevos del día** (el primer día todos son nuevos y aún no se habla de bajas de precio).

La celda siguiente aplica la regla **a los datos de hoy**:

In [ ]:
UMBRAL, MIN_COMP = 0.85, 5

def banda_km(km):
    if pd.isna(km): return None
    return "0-25k" if km <= 25_000 else "25-50k" if km <= 50_000 else "50-100k" if km <= 100_000 else ">100k"

v = vista.copy()
v["precio_valido"] = v["precio"].between(20e6, 400e6) & v["moneda"].eq("PESO")
v["banda_km"] = v["kilometraje"].apply(banda_km)
validos = v[v["precio_valido"]]

NIVELES = [
    ("mismo año, combustible y banda de km", lambda f, o: (o.anio == f.anio) & (o.combustible == f.combustible) & (o.banda_km == f.banda_km)),
    ("mismo año y combustible",              lambda f, o: (o.anio == f.anio) & (o.combustible == f.combustible)),
    ("año ±1 y mismo combustible",           lambda f, o: ((o.anio - f.anio).abs() <= 1) & (o.combustible == f.combustible)),
]

def evaluar(f):
    otros = validos[validos.id_anuncio != f.id_anuncio]
    for nombre, cond in NIVELES:
        grupo = otros[cond(f, otros)]
        if len(grupo) >= MIN_COMP:
            med, q1 = grupo.precio.median(), grupo.precio.quantile(0.25)
            ganga = f.precio <= UMBRAL * med and f.precio < q1
            pct = 1 - f.precio / med
            km_txt = f"{f.kilometraje:,.0f}".replace(",", ".")
            expl = (f"Corolla {f.anio} {str(f.combustible).lower()}, {km_txt} km: ${f.precio/1e6:,.1f} M "
                    f"frente a una mediana de ${med/1e6:,.1f} M en {len(grupo)} comparables ({nombre}); "
                    f"{pct:.0%} {'por debajo' if pct > 0 else 'por encima'}, Q1 = ${q1/1e6:,.1f} M.")
            return pd.Series({"referencia": nombre, "n_comparables": len(grupo), "mediana_ref": med,
                              "q1_ref": q1, "posible_ganga": ganga, "explicacion": expl})
    return pd.Series({"referencia": "datos insuficientes", "n_comparables": 0, "mediana_ref": None,
                      "q1_ref": None, "posible_ganga": False,
                      "explicacion": "No hay al menos 5 comparables: no se puede evaluar."})

res = pd.concat([validos.reset_index(drop=True), validos.reset_index(drop=True).apply(evaluar, axis=1)], axis=1)
print("Precios inválidos excluidos:", (~v.precio_valido).sum())
print(res["referencia"].value_counts(), "\n")
gangas = res[res.posible_ganga]
print(f"Posibles gangas hoy: {len(gangas)}")
for e in gangas.explicacion: print(" •", e)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
(validos.precio / 1e6).plot.hist(bins=15, ax=ax[0], edgecolor="white")
ax[0].set(title="Distribución de precios (hoy)", xlabel="Precio (millones COP)", ylabel="Anuncios")
validos.assign(precio_m=validos.precio / 1e6).boxplot(column="precio_m", by="anio", ax=ax[1])
ax[1].set(title="Precio por año del modelo", xlabel="Año", ylabel="Millones COP"); plt.suptitle("")
plt.tight_layout(); plt.show()
(validos.precio / 1e6).describe(percentiles=[.25, .5, .75]).round(1).rename("precio (M COP)")

## 8. Punto 7 — Correo diario (5 %)
Plantilla HTML, enviada por Gmail SMTP a **ja.acevedor1@uniandes.edu.co** y **dl.reyest@uniandes.edu.co**:
1. **Encabezado:** fecha del reporte y estado general (✅ OK / ⚠️ parcial / ❌ falló), leído del log de las 8:00. Si la ejecución sigue en curso o no existe, lo dice.
2. **Cifras:** enlaces encontrados, nuevos, descargas exitosas y fallidas.
3. **Errores relevantes** y etapas pendientes.
4. **Gráfica** incrustada + resumen (n, mediana, cuartiles).
5. **Gangas nuevas del día:** enlace, características, precio, referencia y explicación — o "No se detectaron posibles gangas hoy".

**Se envía siempre**, aunque no haya nuevos, no haya gangas o falle parte del proceso.

## 9. Punto 8 — Entrega (10 %)
Zip con: código + `requirements.txt` + README · `datos/diarios/` (6 días) · `corolla.db` + CSV exportados · `logs/log_ejecuciones.csv` · los 6 correos exportados a PDF · documento de 1 página con el diagrama.

## ✅ Decisiones para revisar juntos
| # | Decisión | Propuesta | ¿De acuerdo? |
|---|---|---|---|
| 1 | Dónde corre | GitHub Actions (ya probado) | |
| 2 | Almacenamiento | SQLite + exportación CSV diaria | |
| 3 | Horario | arranca 20 min antes y espera la hora exacta | |
| 4 | Umbral de ganga | 15 % bajo la mediana **y** bajo Q1, mínimo 5 comparables | |
| 5 | Correo emisor | Gmail con contraseña de aplicación (secret del repo) | |
| 6 | Reparto de trabajo | por definir | |